In [ ]:
!pip install openai==0.28

In [ ]:
!pip install sentence-transformers faiss-cpu

In [ ]:
!pip install rank_bm25

In [ ]:
!pip install pandas

# Importing Dependencies

In [4]:
import openai
import pandas as pd
import re
import time
import random
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import json

In [ ]:
openai.api_key = "API KEY"

# Datasets

In [6]:
df = pd.read_csv("../Code_Snippets.csv")

In [7]:
df.head()

,ID,Problem Title,Problem Description,Submitted Code,Problem Type,Source
0,1,Find All K-Distant Indices in an Array,You are given a 0-indexed integer array nums a...,class Solution:\r\n def findKDistantIndices...,Easy,LeetCode
1,2,Two Sum,Given an array of integers nums and an integer...,"class Solution:\r\n def twoSum(self, nums: ...",Easy,LeetCode
2,3,Symmetric Tree,"Given the root of a binary tree, check whether...","class Solution:\r\n def isSymmetric(self, r...",Easy,LeetCode
3,4,Same Tree,"Given the roots of two binary trees p and q, w...","class Solution:\r\n def isSameTree(self, p:...",Easy,LeetCode
4,5,Binary Tree Inorder Traversal,"Given the root of a binary tree, return the in...",class Solution:\r\n def inorderTraversal(se...,Easy,LeetCode


In [8]:
df.shape

(300, 6)

In [9]:
human_eval_df = pd.read_csv("../rag_data/test.csv")

In [10]:
human_eval_df.shape

(164, 5)

In [11]:
file_path = '../rag_data/mbpp.jsonl'
data = []
with open(file_path, 'r') as file:
    for line in file:
        data.append(json.loads(line))

mbpp_df = pd.DataFrame(data)
mbpp_df.shape

(974, 6)

In [12]:
task_descriptions = (df['Problem Title'] + " " + df['Problem Description']).tolist()
code_snippets = df['Submitted Code'].tolist()

# RAG (SBERT_LSH_Prompt2)

In [14]:
human_eval_df["prompt"] = human_eval_df["prompt"].astype(str)
human_eval_df["canonical_solution"] = human_eval_df["canonical_solution"].astype(str)
human_eval_df["test"] = human_eval_df["test"].astype(str)

mbpp_df["text"] = mbpp_df["text"].astype(str)
mbpp_df["code"] = mbpp_df["code"].astype(str)
mbpp_df["test_list"] = mbpp_df["test_list"].astype(str)

In [15]:
all_tasks = human_eval_df['prompt'].tolist() + mbpp_df['text'].tolist()
all_codes = human_eval_df['canonical_solution'].tolist() + mbpp_df['code'].tolist()
all_tests = human_eval_df['test'].tolist() + mbpp_df['test_list'].tolist()

In [16]:
# 1. Load SBERT model
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')  # Fast and lightweight

In [21]:
# Batch encode all tasks
print("Encoding all task descriptions...")
all_task_embeddings = sbert_model.encode(all_tasks, batch_size=32, show_progress_bar=True, convert_to_numpy=True)

Encoding all task descriptions...


Batches:   0%|          | 0/36 [00:00<?, ?it/s]

In [22]:
# Normalize embeddings for cosine similarity
faiss.normalize_L2(all_task_embeddings)

In [23]:
# Create FAISS index using cosine similarity
dimension = all_task_embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dimension)
faiss_index.add(all_task_embeddings)

In [24]:
# Store (task, code, test) for retrieval
example_store = list(zip(all_tasks, all_codes, all_tests))

In [25]:
def retrieve_examples(task_description, top_k=3):
    # Encode and normalize the query
    query_embedding = sbert_model.encode([task_description], convert_to_numpy=True)
    faiss.normalize_L2(query_embedding)

    # Perform similarity search
    _, indices = faiss_index.search(query_embedding, top_k)

    # Format retrieved examples
    retrieved = []
    for idx in indices[0]:
        if 0 <= idx < len(example_store):
            task, code, test = example_store[idx]
            example = f"# Task:\n{task}\n\n# Solution:\n{code}\n\n# Test:\n{test}"
            retrieved.append(example)
    return retrieved

In [27]:
MAX_RETRIES = 7
BASE_DELAY = 7
start_index = 0
data3 = []

In [28]:
for i, (code_snippet, task_description) in enumerate(zip(code_snippets[start_index:], task_descriptions[start_index:]), start=start_index + 1):
    retries = 0
    success = False

    # Retrieve examples using SBERT_LSH
    retrieved_examples = retrieve_examples(task_description, top_k=3)
    retrieved_text = "\n\n".join(retrieved_examples)

    while retries < MAX_RETRIES:
        try:
            # Generate response using OpenAI API
            response = openai.ChatCompletion.create(
                model="gpt-4o",
                messages=[
                    {
                        "role": "system",
                        "content": f"""You are a helpful AI assistant working with developers to create unit test cases from codes
                        to help the development process. You have access to a retrieval system that provides relevant past examples
                        to assist in generating high-quality test cases."""
                    },
                    {
                        "role": "user",
                        "content": f"""You are given a Python function along with its task description. Your goal is to generate
                    a complete and executable set of test cases for the function. The test cases should be written using Python
                    using the unittest framework and should comprehensively cover various scenarios, including:
                    * Typical cases (common expected inputs).
                    * Edge cases (unusual but valid inputs).
                    * Boundary conditions (extreme values or limits).
                    The output should be a fully standalone Python script that can be executed without modification. Ensure that the
                    generated test script includes:
                    (1) A properly structured TestSolution class with multiple test_ methods.
                    (2) Descriptive test method names that clearly indicate the scenario being tested.
                    (3) Assertions validating the correctness of the function's output.
                    (4) The function should be encapsulated within a Solution class for better organization.

                    Important Requirements:
                    * Include the if __name__ == '__main__': unittest.main() block to allow direct execution.
                    * Do not provide any explanations—only return the complete test script.

                    Here are some relevant test case examples retrieved from a similar problem to improve quality:
                        {retrieved_text}

                    Here are some examples:

                        Example 1:
                        -------------------------
                        Task Description: Write a function that returns the factorial of a given number.

                        Code Snippet:
                        def factorial(n):
                            if n == 0 or n == 1:
                                return 1
                            return n * factorial(n - 1)
    
                        Output:
                        ```python
                        import unittest

                        class Solution:
                            def factorial(self, n):
                                if n == 0 or n == 1:
                                    return 1
                                return n * self.factorial(n - 1)
                        
                        class TestSolution(unittest.TestCase):
                            def setUp(self):
                                self.solution = Solution()
                        
                            def test_factorial_of_0(self):
                                self.assertEqual(self.solution.factorial(0), 1)
                        
                            def test_factorial_of_1(self):
                                self.assertEqual(self.solution.factorial(1), 1)
                        
                            def test_factorial_of_3(self):
                                self.assertEqual(self.solution.factorial(3), 6)
                        
                            def test_factorial_of_5(self):
                                self.assertEqual(self.solution.factorial(5), 120)
                        
                            def test_factorial_of_7(self):
                                self.assertEqual(self.solution.factorial(7), 5040)
                        
                            def test_factorial_of_10(self):
                                self.assertEqual(self.solution.factorial(10), 3628800)
                        
                            def test_factorial_of_12(self):
                                self.assertEqual(self.solution.factorial(12), 479001600)
                        
                        if __name__ == '__main__':
                            unittest.main()
                        ```

                        Example 2:
                    -------------------------
                    Task Description: Write a function that reverses a string.

                    Code Snippet:
                    def reverse_string(s):
                        return s[::-1]

                    Output:
                    ```python
                    import unittest

                    class Solution:
                        def reverse_string(self, s):
                            return s[::-1]
                    
                    class TestSolution(unittest.TestCase):
                        def setUp(self):
                            self.solution = Solution()
                    
                        def test_reverse_string_hello(self):
                            self.assertEqual(self.solution.reverse_string("hello"), "olleh")
                    
                        def test_reverse_string_world(self):
                            self.assertEqual(self.solution.reverse_string("world"), "dlrow")
                    
                        def test_reverse_string_empty(self):
                            self.assertEqual(self.solution.reverse_string(""), "")
                    
                        def test_reverse_string_single_char(self):
                            self.assertEqual(self.solution.reverse_string("a"), "a")
                    
                        def test_reverse_string_special_characters(self):
                            self.assertEqual(self.solution.reverse_string("123@abc"), "cba@321")
                    
                        def test_reverse_string_mixed_case(self):
                            self.assertEqual(self.solution.reverse_string("AbCDeF"), "FeDCbA")
                    
                        def test_reverse_string_with_spaces(self):
                            self.assertEqual(self.solution.reverse_string("Open AI"), "IA nepO")
                    
                    if __name__ == '__main__':
                        unittest.main()
                    ```

                    Now generate the test cases for the following function:

                    Task Description: {task_description}

                    Code Snippet:
                    {code_snippet}

                    Output:
                    Test Script: [generate ONLY the test script]
                        """
                    }
                ],
                temperature=0.7
            )
            data3.append({'Code Snippet': code_snippet, 'Response': response['choices'][0]['message']['content'].strip()})
            print(f"{i}. Processed successfully")
            print(response['choices'][0]['message']['content'].strip())
            time.sleep(BASE_DELAY)  # Apply base delay after success
            success = True
            break

        except Exception as e:
            print(f"Error processing snippet {i}, attempt {retries + 1}: {e}")
            retries += 1
            if "429" in str(e):  # Rate limit error
                delay = BASE_DELAY * (2 ** retries) + random.uniform(1, 3)
                print(f"Rate limit hit. Retrying in {delay:.2f} seconds...")
                time.sleep(delay)
            else:
                break  # Break for non-rate-limit errors

    if not success:
        data3.append({'Code Snippet': code_snippet, 'Response': "Cannot generate"})
        print(f"Failed to process snippet {i}. Defaulted to 'Cannot generate'.")

1. Processed successfully
```python
import unittest

class Solution:
    def findKDistantIndices(self, nums, key, k):
        index = []
        n = len(nums)
        for i in range(n):
            for j in range(n):
                if nums[j] == key and abs(i - j) <= k:
                    index.append(i)
                    break
        return index

class TestSolution(unittest.TestCase):
    def setUp(self):
        self.solution = Solution()

    def test_example_1(self):
        self.assertEqual(self.solution.findKDistantIndices([3,4,9,1,3,9,5], 9, 1), [1,2,3,4,5,6])

    def test_example_2(self):
        self.assertEqual(self.solution.findKDistantIndices([2,2,2,2,2], 2, 2), [0,1,2,3,4])

    def test_no_k_distant_indices(self):
        self.assertEqual(self.solution.findKDistantIndices([1,2,3,4,5], 6, 1), [])

    def test_all_indices_k_distant(self):
        self.assertEqual(self.solution.findKDistantIndices([1,1,1,1,1], 1, 4), [0,1,2,3,4])

    def test_single_element_array(se

In [29]:
output_df3 = pd.DataFrame(data3)
output_df3.to_csv("SBERT_LSH_Prompt2.csv", index=False)
print("Test cases saved to 'SBERT_LSH_Prompt2.csv'")

Test cases saved to 'SBERT_LSH_Prompt2.csv'


In [ ]:
pip install sentence-transformers hnswlib